# 02 Kalshi CPI Market Data Pull

## Objective:

Pull CPI related contracts from Kalshi markets and store them as a clean dataset

**This notebook:**
- Connects to the Kalshi API 
- Discovers CPI markets on Kalshi
- Pulls contract level price snapshots
- Saves them to /data/

In [12]:
import os, time, json
import pandas as pd
import numpy as np
import requests
from datetime import datetime, timezone

BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"
SESSION = requests.Session()
SESSION.headers.update({"accept": "application/json"})

In [13]:
# Helper: pagination

def get_markets_for_series(series_ticker: str, status="open", limit=1000, max_pages=20):
    out = []
    cursor = None
    for _ in range(max_pages):
        params = {"series_ticker": series_ticker, "status": status, "limit": limit}
        if cursor:
            params["cursor"] = cursor
        r = requests.get(f"{BASE_URL}/markets", params=params, timeout=30)
        r.raise_for_status()
        j = r.json()
        out.extend(j.get("markets", []))
        cursor = j.get("cursor") or None
        if not cursor:
            break
    return out

In [14]:
SERIES = "KXECONSTATCPIYOY"
mk_cpi = pd.DataFrame(get_markets_for_series(SERIES, status="open"))

CANON_COLS = [
    "ticker","event_ticker","series_ticker",
    "title","subtitle",
    "close_time","expected_expiration_time","expiration_time",
    "strike_type","custom_strike","cap_strike","floor_strike",
    "last_price","yes_bid_dollars","yes_ask_dollars","no_bid_dollars","no_ask_dollars",
]
keep = [c for c in CANON_COLS if c in mk_cpi.columns]
mk_meta = mk_cpi[keep].copy()

mk_meta["pulled_series"] = SERIES
mk_meta["pulled_at_utc"] = datetime.now(timezone.utc).isoformat()

out_path = "../data/kalshi_cpi_markets_raw.parquet"
mk_meta.to_parquet(out_path, index=False)
print(f"Saved {out_path} with {len(mk_meta)} rows and {mk_meta['event_ticker'].nunique()} events.")

Saved ../data/kalshi_cpi_markets_raw.parquet with 134 rows and 7 events.


- I actually re-coded this step maybe 15 times because I just couldn't get my logic to find CPI categories.
- Chatgpt saved me and told me to stop relying on /series, I don't know why I didn't think of using tickers in the beginning lmao

In [15]:
#Pull and save CPI market metadata

SERIES = "KXECONSTATCPIYOY"

# 1) Pull CPI markets (open)
markets = get_markets_for_series(SERIES, status="open", limit=1000, max_pages=50)
mk = pd.DataFrame(markets)

print(f"Pulled {len(mk)} markets for series={SERIES}")
display_cols = [c for c in ["ticker", "title", "subtitle", "event_ticker", "close_time"] if c in mk.columns]
display(mk[display_cols].head(10))

# 2) Keep spine columns

CANON_COLS = [
    # IDs
    "ticker", "event_ticker", "series_ticker",
    # Labels
    "title", "subtitle",
    # Timing
    "close_time", "expected_expiration_time", "expiration_time",
    # Bucket/strike metadata
    "strike_type", "custom_strike", "cap_strike", "floor_strike",
    # Optional pricing fields (nice to keep if present)
    "last_price", "yes_bid_dollars", "yes_ask_dollars", "no_bid_dollars", "no_ask_dollars",
]
keep = [c for c in CANON_COLS if c in mk.columns]
print(mk.columns)
mk_meta = mk[keep].copy()

# 3) Add snapshot metadata
mk_meta["pulled_series"] = SERIES
mk_meta["pulled_at_utc"] = datetime.now(timezone.utc).isoformat()

# 4) Minimal sanity checks
assert mk_meta["ticker"].notna().all(), "Some markets are missing tickers."
assert mk_meta["event_ticker"].notna().all(), "Some markets are missing event_ticker."
assert mk_meta["ticker"].is_unique, "Duplicate market tickers detected (unexpected)."

print("Unique events:", mk_meta["event_ticker"].nunique())
print("Columns saved:", mk_meta.columns.tolist())

# Keep only markets that have not closed yet.
if "close_time" in mk_meta.columns:
    mk_meta["close_time"] = pd.to_datetime(mk_meta["close_time"], utc=True, errors="coerce")
    now_utc = datetime.now(timezone.utc)
    mk_meta = mk_meta[mk_meta["close_time"].notna() & (mk_meta["close_time"] > now_utc)].copy()

# 5) Save artifact
out_path = "../data/kalshi_cpi_markets_open.parquet"
mk_meta.to_parquet(out_path, index=False)
print(f"Saved: {out_path}")


Pulled 134 markets for series=KXECONSTATCPIYOY


,ticker,title,event_ticker,close_time
0,KXECONSTATCPIYOY-26APR-T4.2,CPI year-over-year in Apr 2026?,KXECONSTATCPIYOY-26APR,2026-05-12T12:29:00Z
1,KXECONSTATCPIYOY-26APR-T4.1,CPI year-over-year in Apr 2026?,KXECONSTATCPIYOY-26APR,2026-05-12T12:29:00Z
2,KXECONSTATCPIYOY-26JUN-T4.0,CPI year-over-year in Jun 2026?,KXECONSTATCPIYOY-26JUN,2026-07-14T12:29:00Z
3,KXECONSTATCPIYOY-26JUN-T3.9,CPI year-over-year in Jun 2026?,KXECONSTATCPIYOY-26JUN,2026-07-14T12:29:00Z
4,KXECONSTATCPIYOY-26JUN-T3.8,CPI year-over-year in Jun 2026?,KXECONSTATCPIYOY-26JUN,2026-07-14T12:29:00Z
5,KXECONSTATCPIYOY-26JUN-T3.7,CPI year-over-year in Jun 2026?,KXECONSTATCPIYOY-26JUN,2026-07-14T12:29:00Z
6,KXECONSTATCPIYOY-26JUN-T3.6,CPI year-over-year in Jun 2026?,KXECONSTATCPIYOY-26JUN,2026-07-14T12:29:00Z
7,KXECONSTATCPIYOY-26MAY-T4.5,CPI year-over-year in May 2026?,KXECONSTATCPIYOY-26MAY,2026-06-10T12:29:00Z
8,KXECONSTATCPIYOY-26MAY-T4.4,CPI year-over-year in May 2026?,KXECONSTATCPIYOY-26MAY,2026-06-10T12:29:00Z
9,KXECONSTATCPIYOY-26MAY-T4.3,CPI year-over-year in May 2026?,KXECONSTATCPIYOY-26MAY,2026-06-10T12:29:00Z


Index(['can_close_early', 'close_time', 'created_time', 'custom_strike',
       'early_close_condition', 'event_ticker', 'expected_expiration_time',
       'expiration_time', 'expiration_value', 'fractional_trading_enabled',
       'last_price_dollars', 'latest_expiration_time', 'liquidity_dollars',
       'market_type', 'no_ask_dollars', 'no_bid_dollars', 'no_sub_title',
       'notional_value_dollars', 'open_interest_fp', 'open_time',
       'previous_price_dollars', 'previous_yes_ask_dollars',
       'previous_yes_bid_dollars', 'price_level_structure', 'price_ranges',
       'response_price_units', 'result', 'rules_primary', 'rules_secondary',
       'settlement_timer_seconds', 'status', 'strike_type', 'tick_size',
       'ticker', 'title', 'updated_time', 'volume_24h_fp', 'volume_fp',
       'yes_ask_dollars', 'yes_ask_size_fp', 'yes_bid_dollars',
       'yes_bid_size_fp', 'yes_sub_title'],
      dtype='object')
Unique events: 7
Columns saved: ['ticker', 'event_ticker', 'title', 'c

In [ ]:
# Pull orderbook snapshots (compute p_market)

mk_meta = pd.read_parquet("../data/kalshi_cpi_markets_open.parquet")
tickers = mk_meta["ticker"].dropna().unique().tolist()

print("Tickers to snapshot:", len(tickers))

def get_orderbook(ticker: str) -> dict:
    url = f"{BASE_URL}/markets/{ticker}/orderbook"
    r = SESSION.get(url, timeout=30)
    r.raise_for_status()
    return r.json()

def best_bid(levels):
    # levels often like [[price, qty], ...]
    if not levels:
        return None, None
    best = max(levels, key=lambda x: x[0])
    return best[0], best[1]

snap_ts = datetime.now(timezone.utc).isoformat()

rows = []
for t in tickers:
    ob = get_orderbook(t)

    yes_levels = (ob.get("orderbook", {}).get("yes") or ob.get("yes") or [])
    no_levels  = (ob.get("orderbook", {}).get("no")  or ob.get("no")  or [])

    yes_px, yes_qty = best_bid(yes_levels)
    no_px,  no_qty  = best_bid(no_levels)

    rows.append({
        "ts_utc": snap_ts,
        "ticker": t,
        "yes_best_bid_px": yes_px,
        "yes_best_bid_qty": yes_qty,
        "no_best_bid_px": no_px,
        "no_best_bid_qty": no_qty,
        # optional: keep raw so you can debug later
        "orderbook_raw": json.dumps(ob),
    })

ob_snap = pd.DataFrame(rows)
display(ob_snap.head(10))

# Save raw snapshot
ob_snap.to_parquet("../data/kalshi_cpi_orderbooks_snapshot.parquet", index=False)
print("Saved ../data/kalshi_cpi_orderbooks_snapshot.parquet")


Tickers to snapshot: 134


In [ ]:
# Compute Market Implied p


# Baseline: interpret YES best bid dollars as probability
# Later we improve to mid-price if ask exists or using both sides.

p_mkt = ob_snap[["ts_utc", "ticker", "yes_best_bid_px"]].copy()
p_mkt = p_mkt.rename(columns={"yes_best_bid_px": "p_market"})

display(p_mkt.head(10))

p_mkt.to_parquet("../data/kalshi_cpi_market_probs_snapshot.parquet", index=False)
print("Saved ../data/kalshi_cpi_market_probs_snapshot.parquet")

,ts_utc,ticker,p_market
0,2026-04-17T07:18:20.867680+00:00,KXECONSTATCPIYOY-26APR-T4.2,None
1,2026-04-17T07:18:20.867680+00:00,KXECONSTATCPIYOY-26APR-T4.1,None
2,2026-04-17T07:18:20.867680+00:00,KXECONSTATCPIYOY-26JUN-T4.0,None
3,2026-04-17T07:18:20.867680+00:00,KXECONSTATCPIYOY-26JUN-T3.9,None
4,2026-04-17T07:18:20.867680+00:00,KXECONSTATCPIYOY-26JUN-T3.8,None
5,2026-04-17T07:18:20.867680+00:00,KXECONSTATCPIYOY-26JUN-T3.7,None
6,2026-04-17T07:18:20.867680+00:00,KXECONSTATCPIYOY-26JUN-T3.6,None
7,2026-04-17T07:18:20.867680+00:00,KXECONSTATCPIYOY-26MAY-T4.5,None
8,2026-04-17T07:18:20.867680+00:00,KXECONSTATCPIYOY-26MAY-T4.4,None
9,2026-04-17T07:18:20.867680+00:00,KXECONSTATCPIYOY-26MAY-T4.3,None


Saved ../data/kalshi_cpi_market_probs_snapshot.parquet
